| Tipo                         | Métrica                                     | ¿Qué mide?                                                   | Adecuación en tu caso                                                          |
| ---------------------------- | ------------------------------------------- | ------------------------------------------------------------ | ------------------------------------------------------------------------------ |
| **Calidad lingüística**      | **BLEU (1–4)**                              | Precisión de *n-gramas*.                                     | Útil como baseline, aunque estricta con sinónimos.                             |
|                              | **ROUGE-L**                                 | Longest Common Subsequence (recall de secuencias).           | Evalúa si el modelo recupera las partes esenciales de la referencia.           |
|                              | **METEOR**                                  | Usa sinónimos, stemming y alineación semántica.              | Muy fiable con una sola referencia.                                            |
|                              | **SPICE**                                   | Evalúa la semántica (objetos, atributos, relaciones).        | Excelente para medir comprensión del significado meteorológico.                |
| **Eficiencia computacional** | **Tiempo de entrenamiento (h/épocas)**      | Tiempo requerido por cada época y total hasta convergencia.  | Clave para comparar modelos en entornos con recursos limitados.                |
|                              | **Costo económico (\$)**                    | Dinero estimado en función de uso de GPU/TPU/CPU en la nube. | Útil si planeas escalar o usar infraestructura de pago.                        |
|                              | **Consumo energético (kWh)**                | Energía consumida durante entrenamiento/inferencia.          | Importante para sostenibilidad e investigación responsable.                    |
|                              | **CO₂ equivalente (kgCO₂e)**                | Estimación de huella de carbono del entrenamiento.           | Cada vez más usada en papers para reportar impacto ambiental.                  |
|                              | **Memoria utilizada (GB)**                  | RAM/VRAM usada por el modelo en entrenamiento e inferencia.  | Útil para determinar escalabilidad a distintos dispositivos.                   |
|                              | **Velocidad de inferencia (ms por imagen)** | Tiempo que tarda el modelo en generar una descripción.       | Crítico si se busca aplicación en tiempo real (ej. observación meteorológica). |


In [1]:
import os
import re
import json
import tempfile
import pandas as pd
import subprocess
from pathlib import Path
from typing import List, Tuple, Optional
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

def _safe_nltk_download(pkg: str):
    try:
        nltk.data.find(pkg)
    except LookupError:
        try:
            nltk.download(pkg.split("/")[-1], quiet=True)
        except Exception:
            pass

_safe_nltk_download("tokenizers/punkt")
_safe_nltk_download("corpora/wordnet")
_safe_nltk_download("corpora/omw-1.4")


from bert_score import score as bert_score

In [2]:
INPUT_PATH         = '../../resources/utils/images_weather_description_test.csv'
OUTPUT_PATH        = '../../resources/utils/metrics_test.csv'
LANGUAGE           = 'en'
COLUM_ORIGINAL     = 'des_original'
COLUMN_DESCRIPTION = 'des_algorithm'  

In [3]:
_PUNCT_RE = re.compile(r"[^\wáéíóúüñçàèìòùäëïöüÁÉÍÓÚÜÑÇÀÈÌÒÙÄËÏÖÜ-]+", re.UNICODE)

In [4]:
def normalize(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip().lower()
    text = _PUNCT_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text)
    return text

def tokenize(text: str) -> List[str]:
    return normalize(text).split()

In [5]:
# -----------------------
# BLEU (1..4) con suavizado
# -----------------------
_SMOOTH = SmoothingFunction().method3

def compute_bleu_1_4(reference: str, hypothesis: str) -> Tuple[float, float, float, float]:
    ref_tok = tokenize(reference)
    hyp_tok = tokenize(hypothesis)
    refs = [ref_tok]
    if not ref_tok or not hyp_tok:
        return (0.0, 0.0, 0.0, 0.0)
    b1 = sentence_bleu(refs, hyp_tok, weights=(1,0,0,0), smoothing_function=_SMOOTH)
    b2 = sentence_bleu(refs, hyp_tok, weights=(0.5,0.5,0,0), smoothing_function=_SMOOTH)
    b3 = sentence_bleu(refs, hyp_tok, weights=(1/3,1/3,1/3,0), smoothing_function=_SMOOTH)
    b4 = sentence_bleu(refs, hyp_tok, weights=(0.25,0.25,0.25,0.25), smoothing_function=_SMOOTH)
    return b1, b2, b3, b4

In [6]:
# -----------------------
# ROUGE-L (LCS) F-score
# -----------------------
def _lcs_len(x: List[str], y: List[str]) -> int:
    m, n = len(x), len(y)
    if m == 0 or n == 0:
        return 0
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m):
        xi = x[i]
        for j in range(n):
            dp[i+1][j+1] = dp[i][j] + 1 if xi == y[j] else max(dp[i][j+1], dp[i+1][j])
    return dp[m][n]

def compute_rouge_l(reference: str, hypothesis: str, beta: float = 1.2) -> float:
    ref = tokenize(reference)
    hyp = tokenize(hypothesis)
    if not ref or not hyp:
        return 0.0
    lcs = _lcs_len(ref, hyp)
    prec = lcs / max(1, len(hyp))
    rec  = lcs / max(1, len(ref))
    if prec == 0 or rec == 0:
        return 0.0
    beta2 = beta * beta
    return ( (1 + beta2) * prec * rec ) / (rec + beta2 * prec)

In [7]:
# -----------------------
# METEOR (NLTK)
# -----------------------
def compute_meteor(reference: str, hypothesis: str) -> float:
    try:
        return float(meteor_score([reference], hypothesis))
    except Exception:
        # Fallback simple si falla WordNet/OMW
        ref_toks = set(tokenize(reference))
        hyp_toks = set(tokenize(hypothesis))
        if not ref_toks or not hyp_toks:
            return 0.0
        inter = len(ref_toks & hyp_toks)
        prec = inter / len(hyp_toks)
        rec  = inter / len(ref_toks)
        return 0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec)

In [8]:
# -----------------------
# BERTScore (usando modelo multilingüe por defecto)
# -----------------------
def compute_bertscore(references: List[str], hypotheses: List[str], lang: str = "es") -> List[float]:
    _, _, F1 = bert_score(hypotheses, references, lang=lang, verbose=False)
    return [f.item() for f in F1]

In [9]:
def evaluate_csv(in_path, out_path, lang, col_org, col_des):
    df = pd.read_csv(in_path)
    required = [col_org, col_des]
    for col in required:
        if col not in df.columns:
            raise ValueError(f"Falta la columna requerida: '{col}'")

    results = []
    # --- métricas clásicas ---
    for _, row in df.iterrows():
        ref_raw = row.get(col_org, "")
        hyp_raw = row.get(col_des, "")

        b1, b2, b3, b4 = compute_bleu_1_4(ref_raw, hyp_raw)
        rl = compute_rouge_l(ref_raw, hyp_raw)
        met = compute_meteor(ref_raw, hyp_raw)

        results.append({
            "BLEU_1": b1,
            "BLEU_2": b2,
            "BLEU_3": b3,
            "BLEU_4": b4,
            "ROUGE_L": rl,
            "METEOR": met
        })

    metrics_df = pd.DataFrame(results)

    # --- BERTScore (F1) ---
    refs = df[col_org].astype(str).tolist()
    hyps = df[col_des].astype(str).tolist()
    berts = compute_bertscore(refs, hyps, lang=lang)
    metrics_df["BERTScore_F1"] = berts

    # --- unir y guardar ---
    out_df = pd.concat([df.reset_index(drop=True), metrics_df], axis=1)
    out_df.to_csv(out_path, index=False)
    
    return out_df

In [10]:
def main():
    in_path  = INPUT_PATH   
    out_path = OUTPUT_PATH  
    lang     = LANGUAGE
    col_org  = COLUM_ORIGINAL
    col_des  = COLUMN_DESCRIPTION
    
    evaluate_csv(in_path, out_path, lang=lang, col_org=col_org, col_des=col_des)

In [11]:
%%time

if __name__ == "__main__":
    main()

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CPU times: total: 44min 54s
Wall time: 11min 32s
